# Products Analysis

#### Products KPIs
- Top & low performers products (Product concentration)
- Category/subcategory performance
- Average order quantity per product
- price vs volume relationship
- Return Rate per Product
- 3 months forcast
- crosselling 

In [123]:
import pandas as pd

import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from plotly.subplots import make_subplots

import numpy as np

In [124]:
clean_transactions=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_transactions.csv")
clean_orders=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_orders.csv")
clean_customers=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_customers.csv")
clean_catalogue=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_catalogue.csv")


In [125]:
Valid_sales=clean_orders[clean_orders["order_status"].isin(["Terminée", "Partiellement remboursée"])]

# Top vs low performance products

In [126]:
def Top_products(clean_transactions, top_n=10):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]
    top_products=(
        ok_transactions.groupby("product_name").agg(
                total_revenue=("line_total", "sum"),
                total_orders=("order_id_stage", "count")
            ).reset_index().sort_values(by="total_revenue", ascending=False).head(top_n)
        )
    return top_products

In [127]:
def low_performers_products(clean_transactions, bottom_n=10):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]
    low_performers_products=(
        ok_transactions.groupby("product_name").agg(
                total_revenue=("line_total", "sum"),
                total_orders=("order_id_stage", "count")
            ).reset_index().sort_values(by="total_revenue", ascending=True).head(bottom_n)
    )
    return low_performers_products

In [128]:
Top_products(clean_transactions, top_n=10)

,product_name,total_revenue,total_orders
59,Chaudière SL-DL32,21576893.72,73
62,Chaudière SL-DM24,16800699.03,128
63,Chaudière SL-DM28,12076085.65,85
60,Chaudière SL-DL36,8664460.00,58
61,Chaudière SL-DM18,8315023.75,82
298,Moniteur 2 fils 4.3 pouces VFE11,6117174.03,75
228,Lampe led éclairage public 50W-6.5K-360°-AC LE...,6113941.64,33
427,Régulateur B25-21mbar,3826784.21,16
57,Chaudière SL-DE36,3483398.00,20
131,Disjoncteur différentiel sensible 30mA 1P+N DD...,2753003.87,19


In [129]:
low_performers_products(clean_transactions, bottom_n=10)

,product_name,total_revenue,total_orders
170,Feuille d'aluminium pour l'isolation WCT,80.0,1
425,Ruban LED Vert 1 ligne 8W,160.0,1
212,Interrupteur sonnette Sabah,200.0,1
377,Prise téléphone Adenium,210.0,1
55,Cartouche Tulip,300.0,1
424,Ruban LED Rouge 1 ligne 8W,320.0,1
85,Connecteur d'alimentation Ruban LED DOB 1 ligne,345.0,1
92,DISJONCTEUR BIPOLAIRE ECB3.BP.20,414.0,1
308,Panneau d'isolation arrière 24kW,450.0,1
309,Panneau d'isolation avant 24kW,450.0,1


## Category/subcategory performance


In [130]:
def performance_per_category(clean_transactions):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]

    top_categories = (
        ok_transactions
        .groupby("category")
        .agg(
            total_revenue=("line_total", "sum"),
            total_orders=("order_id_stage", "nunique")
        )
        .reset_index()
        .sort_values(by="total_revenue", ascending=False)
    )
    return top_categories
    

In [131]:
performance_per_category(clean_transactions)

,category,total_revenue,total_orders
2,Sanitaire,1.259761e+08,2996
3,Électricité,5.443734e+07,3038
0,Autre,9.543930e+06,789
1,Menuiserie,4.138513e+05,74


In [132]:
def performance_per_subcategory(clean_transactions):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]
    ok_transactions = ok_transactions.drop(columns=["category", "subcategory"])

    merged = ok_transactions.merge(
        clean_catalogue[["sku", "category", "subcategory"]],
        on="sku",
        how="left"
    )
    top_subcategory = (
        merged
        .groupby("subcategory")
        .agg(
            total_revenue=("line_total", "sum"),
            total_orders=("order_id_stage", "nunique")
        )
        .reset_index()
        .sort_values(by="total_revenue", ascending=False)
    )
    return top_subcategory
    

In [133]:
performance_per_subcategory(clean_transactions)

,subcategory,total_revenue,total_orders
3,Chauffage et régulation gaz,87628807.35,698
14,Tableau électrique et protection,22559720.20,1669
1,Alarme sécurité de la maison,20363206.04,657
13,Robinetterie sanitaire,19235008.99,868
8,Pièces de rechange pour S.A.V.,19024962.81,1792
7,Luminaire et éclairage LED,6624366.43,137
11,Robinets gaz et accessoires,4639558.62,71
6,Interrupteurs et prises,3621392.32,629
2,Baignoires,3060424.00,18
12,Robinets multicouches et accessoires,1499922.07,64


## Average order quantity per product


In [134]:
def avg_order_quant_per_product(clean_transactions):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]

    avg_quantities = (
        ok_transactions
        .groupby("product_name")
        .agg(
            total_units_sold=("quantity", "sum"),
            avg_quantity_per_order=("quantity", "mean"),
            number_of_orders=("order_id_stage", "nunique")
        )
        .reset_index()
    )
    return avg_quantities

In [135]:
avg_order_quant_per_product(clean_transactions)

,product_name,total_units_sold,avg_quantity_per_order,number_of_orders
0,Afficheur pour chauffe-eau à T° const,80.0,1.379310,58
1,Applique murale femelle 16×1/2″,17.0,17.000000,1
2,Applique murale mâle 16×3/4″,112.0,22.400000,5
3,Assemblage de la vanne d'entrée d'eau 14L-28kW,1.0,1.000000,1
4,Auto Eclairage NOOR,57.0,4.750000,12
...,...,...,...,...
479,Ventilateur 45W,146.0,2.281250,64
480,Ventilateur 56W,91.0,1.750000,52
481,Ventilateur 67W,20.0,2.222222,9
482,Ventilateur 8W pour chauffe-eau à T° const WCT...,95.0,1.583333,60


## price vs volume relationship


In [136]:
def price_vs_volume(clean_transactions):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]
    price_volume = (
        ok_transactions
        .groupby("product_name")
        .agg(
            total_units_sold=("quantity", "sum"),
            avg_price=("unit_price", "mean"),
            total_revenue=("line_total", "sum")
        )
        .reset_index()
    )
    return price_volume

In [137]:
price_vs_volume(clean_transactions)

,product_name,total_units_sold,avg_price,total_revenue
0,Afficheur pour chauffe-eau à T° const,80.0,1381.310345,115471.00
1,Applique murale femelle 16×1/2″,17.0,555.000000,9435.00
2,Applique murale mâle 16×3/4″,112.0,621.958000,67293.70
3,Assemblage de la vanne d'entrée d'eau 14L-28kW,1.0,3360.000000,3360.00
4,Auto Eclairage NOOR,57.0,804.310000,45966.72
...,...,...,...,...
479,Ventilateur 45W,146.0,5329.593750,804691.94
480,Ventilateur 56W,91.0,4915.523462,451873.08
481,Ventilateur 67W,20.0,5874.444444,116870.00
482,Ventilateur 8W pour chauffe-eau à T° const WCT...,95.0,1434.750000,132700.00


## Graphs

In [138]:
topproducts= Top_products(clean_transactions, top_n=10)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("total_revenue", "total_orders")
)

fig.add_trace(
    go.Bar(x=topproducts["product_name"], y=topproducts["total_revenue"], name="total_revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=topproducts["product_name"], y=topproducts["total_orders"], name="total_orders"),
    row=1, col=2
)
fig.update_layout(title_text="Top Products", showlegend=False)
fig.show()

In [139]:
low_performers_pro=low_performers_products(clean_transactions, bottom_n=10)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("total_revenue", "total_orders")
)

fig.add_trace(
    go.Bar(x=low_performers_pro["product_name"], y=low_performers_pro["total_revenue"], name="total_revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=topproducts["product_name"], y=topproducts["total_orders"], name="total_orders"),
    row=1, col=2
)
fig.update_layout(title_text="Top Products", showlegend=False)
fig.show()